# This is just the definition of the plotting routine. Execute the cell and pass to the next.

In [ ]:
import os
import h5py
import numpy as np
import matplotlib.pyplot as plt
from taurex.binning import FluxBinner

#Options for interactive plotting in Jupyter notebooks 
#%matplotlib widget
plt.style.use('default')

def create_grid_res(resolution, wave_min, wave_max):
    wave_list = []
    width_list = []
    wave = wave_min
    width = wave/resolution    
    
    while wave < wave_max:
        width = wave / (resolution - 0.5) + width/2/(resolution - 0.5)
        wave = resolution * width 
        width_list.append(width)
        wave_list.append(wave)

    return np.array((wave_list ,width_list)).T


def plot_opacity_cross_sections(folder_path,
                                p_val=1,
                                t_val=500,
                                xlim=(0.5, 20),
                                ylim=(1e-35, 1e-16),
                                figsize=(20, 12),
                                lw=0.7,
                                alpha=0.7,
                                logscale=True,
                                filter_molecules=None, binning_resolution=None):
    """
    Plot opacity cross-sections for all (or selected) molecules in a folder at given pressure and temperature.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing TauREx-formatted HDF5 cross-section files.
    p_val : float, optional
        Pressure value in bar to find the closest match in the grid.
    t_val : float, optional
        Temperature value in K to find the closest match in the grid.
    wavelength_unit : str, optional
        Unit for x-axis, either 'nm' or 'micron'.
    xlim : tuple, optional
        Tuple of (min, max) x-axis limits.
    ylim : tuple, optional
        Tuple of (min, max) y-axis limits.
    figsize : tuple, optional
        Figure size.
    lw : float, optional
        Line width.
    alpha : float, optional
        Transparency.
    logscale : bool, optional
        Whether to use logarithmic axes.
    filter_molecules : list of str, optional
        List of molecule names to include. If None, include all.
    """

    plt.figure(figsize=figsize, tight_layout=True)
    first = True
    list_mols = []
    for filename in sorted(os.listdir(folder_path)):
        if not filename.endswith(".h5"):
            continue

        filepath = os.path.join(folder_path, filename)

        with h5py.File(filepath, 'r') as f:
            # Get molecule name
            mol_name_raw = f['mol_name'][()]
            if isinstance(mol_name_raw, np.ndarray):
                mol_name_raw = mol_name_raw[0]
            mol_name = mol_name_raw.decode() if isinstance(mol_name_raw, bytes) else str(mol_name_raw)
            list_mols.append(mol_name)

            if filter_molecules is not None and mol_name not in filter_molecules:
                continue

            # Find nearest temperature and pressure indices
            temp_grid = f['t'][()]
            pres_grid = f['p'][()]

            t_index = np.abs(temp_grid - t_val).argmin()
            p_index = np.abs(pres_grid - p_val).argmin()

            print(f"Processing {mol_name} from {filename} at T={temp_grid[t_index]} K and P={pres_grid[p_index]} bar")

            # Load data
            bin_edges = f['bin_edges'][()]
            xsec = f['xsecarr'][p_index, t_index, :]
            wavelength = 10000 / bin_edges
            if binning_resolution is not None:
                binned_grid = create_grid_res(binning_resolution, wavelength.min(), wavelength.max())[:,0]
                binned_data = FluxBinner(binned_grid).bindown(wavelength, xsec)
                wavelength, xsec = binned_data[0], binned_data[1]
                
            plt.plot(wavelength, xsec, label=filename[:2] + mol_name, lw=lw, alpha=alpha)

            if first:
                actual_T = temp_grid[t_index]
                actual_P = pres_grid[p_index]
                print(f"Temperature grid: {temp_grid}")
                print(f"Pressure grid: {pres_grid}")
                first = False

    print(f"List of available molecules: {list_mols}")
    if first:
        raise RuntimeError("No molecules matched the filters or data could not be read.")

    plt.xlabel('Wavelength (μm)')
    plt.ylabel("Cross Section (cm$^2$/molecule)")
    plt.title(f"Cross Section at {actual_T:.0f} K and {actual_P:.2e} bar")
    if logscale:
        plt.xscale("log")
        plt.yscale("log")
    plt.xlim(*xlim)
    plt.xlim(1, 15)
    plt.ylim(*ylim)
    plt.ylim(1e-25, 1e-18)
    plt.legend(fontsize = 14, frameon=False)
    plt.show()

In [ ]:
plot_opacity_cross_sections(
    folder_path="/path/to/your/cross/section/folder/",
    p_val=40,
    t_val=2000,
    filter_molecules=["CO"]  , 
    figsize = (11, 7),
    binning_resolution=1000
)

# You can add a plot below (or above) of your data to compare it with the cross sections.